# CPU-only final-token accounting correction

This self-contained notebook tests an **opt-in** correction for two pinned public Duck finalizers. It bills the final analyzer tokens that produced no action and reconciles a late worker after the outer cancellation path has already closed its game. Installation is refused when either method's normalized source fingerprint is unknown or wrapped by other code.

The scoring notebook and model are untouched. No GPU, network, model load, game action or competition submission is performed. CPU fixtures use real source-derived public GameRun, Game.finish_game and solver cancellation methods with a no-op environment close. The tests are synthetic software regressions, **not gameplay or score evidence**.

Executed outputs were captured with standard-library `compile`/`exec` in notebook order and one shared namespace. The same cells can run in a Python notebook. Third-party public fixture attribution and hashes are recorded below; no account submissions, quota data, private port source or private benchmark results are included.


In [1]:
import ast, asyncio, contextlib, hashlib, io, json, sys, tempfile, types, unittest
from pathlib import Path

PROVENANCE = {'public_fixture_repository': 'https://github.com/acco-cyber/ARC-AGI-3', 'public_fixture_commit': '58a67ca1c7ddf0d516c7a9376d9f913f9ff5bc60', 'upstream': 'https://github.com/Tufalabs/duck-harness', 'solver_fixture_sha256': 'e22d1425b6dfe5143f6984d8ccfd21fd4435dccf2859a6985d92d985cb91c14e', 'game_fixture_sha256': '8609e0646e96616decc936daf4bff102afd480beb69ef1e9e778cd2c1c7df225', 'embedded_solver_excerpt_sha256': '1989eee41a17dad5801a791b53dc6e7b57b4efe2b61f66bc968dd43152f57e79', 'embedded_game_excerpt_sha256': '4caf364fb04ac5a15d8b1ab3c6cfe30639069dc89251955e945bf3ceede98554', 'helper_sha256': 'ccf3813d64853ab893eab7c763eb3a13774204d33bb39ff3d3cb9c04968c93f5'}
HELPER_SOURCE = '"""Opt-in final-token accounting for two pinned public Duck finalizers.\n\nNo model, request, game action, scoring formula or deadline is changed. This is\nCPU-tested support code, not a score improvement. Install only before play.\nIf an externally cancelled worker never returns, its remaining tokens cannot\nbe recovered here. A later serialized benchmark must be saved after workers\nreturn to include their reconciliation.\n"""\nfrom __future__ import annotations\n\nimport ast\nfrom functools import wraps\nimport hashlib\nimport inspect\nimport textwrap\nimport threading\nimport time\n\n_SESSION_FINGERPRINTS = {\n    "12b11f2456edef277324e9f598b6d1aa43f072805988b3ab42ad3ef851dc1a37",\n    "6e9d39bc9a46b5cc242a33b418a0d73224d10d98f07855bb377ad80b46ad19e8",\n}\n_REMAINING_FINGERPRINT = "12b48d351b7e3a756457f9601e5b4a5f041eb21dfd79fe205fb3611da4c84c03"\n\n\ndef _fingerprint(method):\n    # inspect.getsource unwraps functools.wraps, which could otherwise hide a\n    # foreign wrapper\'s changed behavior behind the pinned original source.\n    if hasattr(method, "__wrapped__"):\n        raise RuntimeError("Wrapped finalizer is not an unmodified pinned method")\n    try:\n        source = textwrap.dedent(inspect.getsource(method))\n        tree = ast.parse(source)\n    except (OSError, TypeError, IndentationError, SyntaxError) as exc:\n        raise RuntimeError("Cannot verify pinned finalizer source") from exc\n    return hashlib.sha256(ast.dump(tree, include_attributes=False).encode()).hexdigest()\n\n\ndef install(solver_module):\n    """Explicitly install once; refuse unrecognized or already modified methods."""\n    if getattr(solver_module, "_arc3_final_tail_accounting_installed", False):\n        return False\n    session_cls = solver_module._HarnessGameSession\n    solver_cls = solver_module.HarnessSolver\n    original_session = session_cls._finish_if_needed\n    original_remaining = solver_cls._finish_remaining\n    if (_fingerprint(original_session) not in _SESSION_FINGERPRINTS\n            or _fingerprint(original_remaining) != _REMAINING_FINGERPRINT):\n        raise RuntimeError("Unrecognized finalizer source; accounting patch not installed")\n    reported_tokens = solver_module._analyzer_reported_tokens\n    if not callable(reported_tokens):\n        raise RuntimeError("Pinned analyzer token accessor is missing")\n    lock = threading.RLock()\n\n    @wraps(original_session)\n    def finish_session(self):\n        with lock:\n            run = self.game.game_run\n            if run is None or getattr(self, "_arc3_tail_accounted_run", None) is run:\n                return\n            if run.final_score is not None:\n                # Only our external-cancellation wrapper may grant this path.\n                # Ordinary finalization remains idempotent even if a counter\n                # changes later. The marker belongs to this exact GameRun.\n                if not getattr(run, "_arc3_external_tail_pending", False):\n                    return\n                current = reported_tokens(self.analyzer)\n                tail = max(0, current - self.token_baseline)\n                run.final_generated_tokens += tail\n                if tail and run.started_at_monotonic is not None:\n                    run.final_wallclock_seconds = max(\n                        run.final_wallclock_seconds,\n                        time.monotonic() - run.started_at_monotonic,\n                    )\n                self.token_baseline = current\n                run._arc3_external_tail_pending = False\n                self._arc3_tail_accounted_run = run\n                return\n            if self.stop_event.is_set() and run.state == "playing":\n                run.state = "cancelled"\n            current = reported_tokens(self.analyzer)\n            tail = max(0, current - self.token_baseline)\n            self.game.finish_game(generated_tokens=tail)\n            self.token_baseline = current\n            self._arc3_tail_accounted_run = run\n\n    @wraps(original_remaining)\n    def finish_remaining(self, games):\n        with lock:\n            # Capture only runs which were unfinished on entry. A normally\n            # completed run must never become eligible for late accounting.\n            pending = [g.game_run for g in games\n                       if g.game_run is not None and g.game_run.final_score is None]\n            original_remaining(self, games)\n            if self._stop_event.is_set():\n                for run in pending:\n                    if run.final_score is not None and run.state == "cancelled":\n                        run._arc3_external_tail_pending = True\n\n    session_cls._finish_if_needed = finish_session\n    solver_cls._finish_remaining = finish_remaining\n    solver_module._arc3_final_tail_accounting_installed = True\n    return True\n'
SOLVER_SOURCE = 'from __future__ import annotations\n\ndef _analyzer_reported_tokens(analyzer: Any) -> int:\n    value = (\n        getattr(analyzer, "generated_tokens", None)\n        if hasattr(analyzer, "generated_tokens")\n        else getattr(analyzer, "total_tokens", 0)\n    )\n    return max(0, int(value or 0))\n\nclass _HarnessGameSession:\n    def _finish_if_needed(self) -> None:\n        run = self.game.game_run\n        if run is not None and run.final_score is None:\n            if self.stop_event.is_set() and run.state == "playing":\n                run.state = "cancelled"\n            # Actions bill the analyzer delta up to token_baseline. Preserve\n            # final deliberation that produced no action, including cancellation\n            # and failure turns, in GameRun.final_generated_tokens.\n            current_tokens = _analyzer_reported_tokens(self.analyzer)\n            generated_tokens = max(0, current_tokens - self.token_baseline)\n            self.game.finish_game(generated_tokens=generated_tokens)\n            self.token_baseline = current_tokens\n\nclass HarnessSolver:\n    async def _run_games(self, games: list[taaf.game.Game]) -> None:\n        self._stop_event.clear()\n        semaphore = asyncio.Semaphore(max(1, int(self.concurrency)))\n        pass_indices_by_game_id: dict[str, int] = {}\n        loop = asyncio.get_running_loop()\n        pool = self._worker_pool\n\n        async def run_one(index: int, pass_index: int, game: taaf.game.Game) -> None:\n            async with semaphore:\n                args = (game, index, pass_index, self._local_server_for_game_index(index))\n                if pool is not None:\n                    await loop.run_in_executor(pool, functools.partial(self._play_one, *args))\n                else:\n                    # _setup wasn\'t called (direct test invocation).\n                    await asyncio.to_thread(self._play_one, *args)\n\n        tasks: list[asyncio.Task[None]] = []\n        for index, game in enumerate(games):\n            game_id = game.game_run.game_id if game.game_run is not None else str(index)\n            pass_index = pass_indices_by_game_id.get(game_id, 0)\n            pass_indices_by_game_id[game_id] = pass_index + 1\n            tasks.append(asyncio.create_task(run_one(index, pass_index, game)))\n        try:\n            await asyncio.gather(\n                *(asyncio.shield(task) for task in tasks), return_exceptions=True\n            )\n        except asyncio.CancelledError:\n            self._stop_event.set()\n            await self._drain_game_tasks(tasks)\n            self._finish_remaining(games)\n            raise\n\n    async def _drain_game_tasks(self, tasks: list[asyncio.Task[None]]) -> None:\n        if not tasks:\n            return\n        timeout = max(0.0, float(self.cancel_drain_timeout_s))\n        if timeout == 0.0:\n            return\n        done, _pending = await asyncio.wait(tasks, timeout=timeout)\n        if done:\n            await asyncio.gather(*done, return_exceptions=True)\n\n    def _finish_remaining(self, games: list[taaf.game.Game]) -> None:\n        for game in games:\n            run = game.game_run\n            if run is not None and run.final_score is None:\n                try:\n                    if self._stop_event.is_set() and run.state == "playing":\n                        run.state = "cancelled"\n                    game.finish_game()\n                except Exception:\n                    pass\n'
GAME_SOURCE = 'from __future__ import annotations\n\n@dataclass(frozen=True)\nclass GameState:\n    """Wraps ``arcengine.FrameDataRaw`` with helper properties (R11.01).\n\n    Fields:\n\n    - ``raw``: the underlying arcengine frame data.\n    - ``previous_action``: the ``ActionInput`` that led here, or ``None``\n      for the initial state.\n    - ``just_won_level``: True if this state crossed a level boundary.\n    """\n\n    raw: arcengine.FrameDataRaw\n    previous_action: arcengine.ActionInput | None = None\n    just_won_level: bool = False\n\n    @property\n    def frame(self) -> Frame:\n        """The final visible frame (last in the raw frame list)."""\n        return Frame(data=self.raw.frame[-1])\n\n    @property\n    def animation_frames(self) -> list[Frame]:\n        """All frames except the final visible one."""\n        return [Frame(data=f) for f in self.raw.frame[:-1]]\n\n    @property\n    def all_frames(self) -> list[Frame]:\n        """All frames (animation + final)."""\n        return [Frame(data=f) for f in self.raw.frame]\n\n    @property\n    def available_actions(self) -> list[int]:\n        """Legal action ids, with RESET (0) always present."""\n        raw = list(self.raw.available_actions)\n        if 0 in raw:\n            return raw\n        return [0, *raw]\n\n    @property\n    def levels_completed(self) -> int:\n        return self.raw.levels_completed\n\n    @property\n    def game_over(self) -> bool:\n        """True if the game has actually ended.\n\n        Synonymous with ``won`` in this framework: arcengine\'s ``GAME_OVER``\n        is a recoverable dead-end-of-attempt (``RESET`` recovers), so only\n        ``WIN`` is a real end state. R11.01 keeps both as separate\n        properties for the conceptual distinction.\n        """\n        return self.raw.state == arcengine.GameState.WIN\n\n    @property\n    def won(self) -> bool:\n        """True if the game ended in a win. Synonym for ``game_over``."""\n        return self.raw.state == arcengine.GameState.WIN\n\n@dataclass(frozen=True)\nclass ActionRecord:\n    """One executed action with token + wallclock cost (R11.02).\n\n    Fields:\n\n    - ``action``: the executed ``ActionInput``.\n    - ``generated_tokens`` / ``uncached_input_tokens``: solver-reported\n      token counts, or 0 if the solver doesn\'t track them (R11.03).\n    - ``wallclock_seconds``: monotonic-clock seconds since\n      ``Game.start_game()``.\n    """\n\n    action: arcengine.ActionInput\n    generated_tokens: int\n    uncached_input_tokens: int\n    wallclock_seconds: float\n\n@dataclass\nclass GameRun:\n    """Per-game run state (R11.02).\n\n    State machine::\n\n        not_started → playing               on Game.start_game()\n        playing     → won                   on execute_action returning won=True\n        playing     → cancelled             on finish_game with asyncio cancel outstanding\n        playing     → gave_up               on finish_game otherwise\n        playing     → crashed               Benchmark teardown, finish_game never called\n\n    Fields:\n\n    - ``game_id`` / ``number_of_levels`` / ``base_actions_per_level`` /\n      ``hint``: basic info populated by ``Game.start_game()``.\n      ``base_actions_per_level`` is ``None`` when the engine hides\n      baselines (e.g. submission mode).\n    - ``state`` / ``levels_completed`` / ``final_score``: progress\n      summary. ``final_score`` is set by ``finish_game()``.\n    - ``history`` / ``actions_per_level``: per-action record and\n      per-level counter. Invariant: ``sum(actions_per_level) == len(history)``.\n    - ``final_generated_tokens`` / ``final_uncached_input_tokens``: tokens\n      the solver spent on its final turn *without* producing a move — the\n      give-up / cancellation decision — reported via ``finish_game(...)``\n      (R11.03). 0 when the solver doesn\'t report them. Counted in token\n      totals but not as an action (``actions_per_level`` is unchanged).\n    - ``final_wallclock_seconds``: monotonic-clock seconds since\n      ``start_game()`` at the moment ``finish_game()`` ran — the same\n      reference and duration semantics as ``ActionRecord.wallclock_seconds``\n      (so it sits on the diagnostics wallclock axis, and survives JSON as a\n      duration). Lets the no-move ``final_*_tokens`` be placed at the time\n      they were actually spent, after the last move.\n    - ``intermediate_states``: full per-step ``GameState`` list. Kept\n      out of JSON per R11.02; persisted to ``intermediate_states.pkl``\n      sidecar.\n    - ``record_intermediate_states``: when false, only history and score\n      bookkeeping are kept. Kaggle submission uses this to avoid storing\n      frames in memory.\n    - ``solver_note`` / ``solver_analysis_html``: solver-provided\n      diagnostics. Either can be re-set during play; the periodic save\n      loop picks up the latest values. ``solver_analysis_html`` is a\n      path **relative to job_dir**, linked from the movie wrapper page.\n    - ``started_at`` / ``started_at_monotonic``: wall-clock start stamps,\n      both populated at ``Game.start_game()``. ``started_at`` is a\n      local-naive ``datetime`` that survives JSON — diagnostics offsets\n      it against ``Benchmark.start_time`` to place each action on a\n      true job-wallclock axis (games queued behind a pool semaphore\n      don\'t collapse to benchmark t=0). ``started_at_monotonic`` is a\n      ``time.monotonic()`` reading used purely as the reference for\n      ``ActionRecord.wallclock_seconds``; immune to NTP / DST jumps\n      mid-run, but not persisted (monotonic epochs don\'t survive a\n      process boundary).\n    """\n\n    game_id: str\n    number_of_levels: int\n    base_actions_per_level: list[int] | None\n    hint: str | None = None\n    state: GameRunState = "not_started"\n    history: list[ActionRecord] = field(default_factory=lambda: list[ActionRecord]())\n    intermediate_states: list[GameState] = field(default_factory=lambda: list[GameState](), repr=False)\n    record_intermediate_states: bool = field(default=True, repr=False)\n    actions_per_level: list[int] = field(default_factory=lambda: list[int]())\n    levels_completed: int = 0\n    started_at: datetime | None = None\n    started_at_monotonic: float | None = field(default=None, repr=False)\n    final_score: float | None = None\n    solver_note: str | None = None\n    solver_analysis_html: str | None = None\n    final_generated_tokens: int = 0\n    final_uncached_input_tokens: int = 0\n    final_wallclock_seconds: float = 0.0\n\n    def __post_init__(self) -> None:\n        if not self.actions_per_level:\n            self.actions_per_level = [0] * self.number_of_levels\n\n    def to_json_dict(self) -> dict[str, Any]:\n        """Serialize to a JSON-friendly dict. ``intermediate_states`` is\n        dropped per R11.02 (persisted to a sidecar pickle instead)."""\n        return {\n            "game_id": self.game_id,\n            "number_of_levels": self.number_of_levels,\n            "base_actions_per_level": self.base_actions_per_level,\n            "hint": self.hint,\n            "state": self.state,\n            "history": [\n                {\n                    "action": {"id": rec.action.id.name, "data": dict(rec.action.data)},\n                    "generated_tokens": rec.generated_tokens,\n                    "uncached_input_tokens": rec.uncached_input_tokens,\n                    "wallclock_seconds": rec.wallclock_seconds,\n                }\n                for rec in self.history\n            ],\n            "record_intermediate_states": self.record_intermediate_states,\n            "actions_per_level": list(self.actions_per_level),\n            "levels_completed": self.levels_completed,\n            "final_score": self.final_score,\n            "solver_note": self.solver_note,\n            "solver_analysis_html": self.solver_analysis_html,\n            "final_generated_tokens": self.final_generated_tokens,\n            "final_uncached_input_tokens": self.final_uncached_input_tokens,\n            "final_wallclock_seconds": self.final_wallclock_seconds,\n            "started_at": self.started_at.isoformat() if self.started_at is not None else None,\n        }\n\n    @classmethod\n    def from_json_dict(cls, d: dict[str, Any]) -> GameRun:\n        started_at_raw = d.get("started_at")\n        run = cls(\n            game_id=d["game_id"],\n            number_of_levels=d["number_of_levels"],\n            base_actions_per_level=d["base_actions_per_level"],\n            hint=d.get("hint"),\n            state=d["state"],\n            actions_per_level=list(d.get("actions_per_level") or []),\n            record_intermediate_states=bool(d.get("record_intermediate_states", True)),\n            levels_completed=d.get("levels_completed", 0),\n            final_score=d.get("final_score"),\n            solver_note=d.get("solver_note"),\n            solver_analysis_html=d.get("solver_analysis_html"),\n            final_generated_tokens=d.get("final_generated_tokens", 0),\n            final_uncached_input_tokens=d.get("final_uncached_input_tokens", 0),\n            final_wallclock_seconds=d.get("final_wallclock_seconds", 0.0),\n            started_at=datetime.fromisoformat(started_at_raw) if started_at_raw else None,\n        )\n        for entry in d["history"]:\n            action = arcengine.ActionInput(\n                id=arcengine.GameAction[entry["action"]["id"]],\n                data=dict(entry["action"]["data"]),\n            )\n            run.history.append(\n                ActionRecord(\n                    action=action,\n                    generated_tokens=entry["generated_tokens"],\n                    uncached_input_tokens=entry["uncached_input_tokens"],\n                    wallclock_seconds=entry["wallclock_seconds"],\n                )\n            )\n        return run\n\n    def _compute_final_score(self) -> float:\n        """ARC-AGI3 score formula. Mirrors\n        ``arc_agi.scorecard.EnvironmentScoreCalculator`` (v0.9.8).\n\n        Per level: ``min(115, (baseline / actions)² × 100)`` if completed\n        with ≥ 1 action, else 0. Weights are 1-indexed (level 0 → 1).\n        Final score is ``total_score / total_weights`` capped at\n        ``max_weights / total_weights × 100``, where ``max_weights``\n        sums only levels that scored > 0.\n        """\n        if self.base_actions_per_level is None or self.number_of_levels == 0:\n            return 0.0\n        total_score = 0.0\n        total_weights = 0\n        max_weights = 0\n        for level_idx in range(self.number_of_levels):\n            weight = level_idx + 1\n            total_weights += weight\n            completed = level_idx < self.levels_completed\n            actions = self.actions_per_level[level_idx] if level_idx < len(self.actions_per_level) else 0\n            baseline = self.base_actions_per_level[level_idx]\n            if completed and actions > 0:\n                level_score = min(115.0, (baseline / actions) ** 2 * 100)\n            else:\n                level_score = 0.0\n            if level_score > 0:\n                max_weights += weight\n            total_score += level_score * weight\n        if total_weights == 0:\n            return 0.0\n        score = total_score / total_weights\n        max_score = max_weights / total_weights * 100\n        return min(score, max_score)\n\nclass Game:\n    def finish_game(self, generated_tokens: int = 0, uncached_input_tokens: int = 0) -> None:\n        """Mark the game as finished (R11.03 / R11.02 state machine).\n\n        If the game is still ``playing``, transitions to ``cancelled``\n        when an asyncio cancel is outstanding on the current task, else\n        ``gave_up``. ``Benchmark.run`` sets the state to ``crashed``\n        before calling this in its teardown path. Idempotent.\n\n        ``generated_tokens`` / ``uncached_input_tokens`` (R11.03) let the\n        solver report tokens it spent on this final turn *without* making a\n        move — the give-up / cancellation decision. They land on the run\'s\n        ``final_*_tokens`` fields and count toward token totals, but not as\n        an action. The finish time is stamped into ``final_wallclock_seconds``\n        (monotonic elapsed since ``start_game``, matching\n        ``ActionRecord.wallclock_seconds``) so those tokens land at the time\n        they were spent on the wallclock axis. Recorded on the first\n        (non-idempotent) call only.\n        """\n        assert self.game_run is not None, "finish_game() before start_game()"\n        if self.game_run.final_score is not None:\n            return\n        self.game_run.final_generated_tokens = generated_tokens\n        self.game_run.final_uncached_input_tokens = uncached_input_tokens\n        if self.game_run.started_at_monotonic is not None:\n            self.game_run.final_wallclock_seconds = time.monotonic() - self.game_run.started_at_monotonic\n        self._finish_game()\n        if self.game_run.state == "playing":\n            cancelling = False\n            try:\n                task = asyncio.current_task()\n                if task is not None and task.cancelling() > 0:\n                    cancelling = True\n            except RuntimeError:\n                # No running loop — sync caller, treat as gave_up.\n                cancelling = False\n            self.game_run.state = "cancelled" if cancelling else "gave_up"\n        self.game_run.final_score = self.game_run._compute_final_score()\n        # One-line per-game finish note to stdout — same fields as the\n        # per-pass row in the diagnostics HTML, plus per-level\n        # actions/baseline so the score is auditable from the log alone.\n        run = self.game_run\n        actions = sum(run.actions_per_level)\n        tokens = sum(rec.generated_tokens for rec in run.history) + run.final_generated_tokens\n        per_level = ",".join(\n            f"{run.actions_per_level[i]}/"\n            f"{run.base_actions_per_level[i] if run.base_actions_per_level is not None else \'?\'}"\n            for i in range(run.number_of_levels)\n        )\n        note_suffix = f\' note="{run.solver_note}"\' if run.solver_note else ""\n        print(\n            f"[finished] {run.game_id} state={run.state} "\n            f"level={run.levels_completed}/{run.number_of_levels} "\n            f"score={run.final_score:.2f} actions={actions} tokens={tokens} "\n            f"per-level={per_level}{note_suffix}"\n        )\n'
FIXTURE_SOURCE = '"""Exercise actual patched solver and TAAF finish methods without GPU packages.\n\nThe two source methods and real GameRun dataclass are loaded by AST so this\naccounting regression can run on a CPU before installing the Kaggle wheelhouse.\nSet ARC3_ACCOUNTING_SOURCE_ROOT to a patched Duck source tree when necessary.\n"""\nfrom __future__ import annotations\n\nimport ast\nimport asyncio\nfrom contextlib import redirect_stdout\nfrom dataclasses import dataclass, field\nimport io\nimport os\nfrom pathlib import Path\nimport threading\nimport time\nfrom types import SimpleNamespace\nimport unittest\n\nSOLVER_PATH = _embedded_solver_path\nGAME_PATH = _embedded_game_path\n\ndef _source_definitions(path, top_names=(), class_methods=None):\n    tree = ast.parse(path.read_text())\n    selected = [ast.ImportFrom(module=\'__future__\', names=[ast.alias(name=\'annotations\')], level=0)]\n    for node in tree.body:\n        if isinstance(node, (ast.ClassDef, ast.FunctionDef)) and node.name in top_names:\n            selected.append(node)\n        if isinstance(node, ast.ClassDef) and class_methods and node.name in class_methods:\n            selected.extend(method for method in node.body\n                            if isinstance(method, ast.FunctionDef) and method.name in class_methods[node.name])\n    module = ast.fix_missing_locations(ast.Module(body=selected, type_ignores=[]))\n    scope = dict(dataclass=dataclass, field=field, asyncio=asyncio, time=time)\n    exec(compile(module, str(path), \'exec\'), scope)\n    return scope\n\n\n_solver = _source_definitions(SOLVER_PATH, (\'_analyzer_reported_tokens\',),\n    {\'_HarnessGameSession\': {\'_finish_if_needed\'}})\n_game = _source_definitions(GAME_PATH, (\'GameState\', \'ActionRecord\', \'GameRun\'), {\'Game\': {\'finish_game\'}})\nGameRun = _game[\'GameRun\']\nfinish_session = _solver[\'_finish_if_needed\']\n\n\nclass MinimalGame:\n    """Use real TAAF finalization/scoring with a no-op environment close."""\n    finish_game = _game[\'finish_game\']\n\n    def __init__(self):\n        self.game_run = GameRun(game_id=\'cpu-test\', number_of_levels=2,\n            base_actions_per_level=[1, 1], state=\'playing\')\n        self.close_count = 0\n\n    def _finish_game(self):\n        self.close_count += 1\n\n\ndef session(*, generated=150, baseline=100, state=\'playing\', stopped=False):\n    game = MinimalGame()\n    game.game_run.state = state\n    stop = threading.Event()\n    if stopped:\n        stop.set()\n    return SimpleNamespace(game=game, analyzer=SimpleNamespace(generated_tokens=generated),\n                           token_baseline=baseline, stop_event=stop)\n\n\ndef finish(subject):\n    with redirect_stdout(io.StringIO()):\n        finish_session(subject)\n\n\n'
TEST_SOURCE = '"""CPU regressions against source-derived public Duck/TAAF finalizers."""\nfrom __future__ import annotations\n\nimport ast\nimport asyncio\nfrom concurrent.futures import ThreadPoolExecutor\nfrom contextlib import redirect_stdout\nimport functools\nimport io\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport textwrap\nimport threading\nfrom types import SimpleNamespace\nimport unittest\n\nfrom final_tail_accounting import install\nimport test_accounting as pinned\n\n\ndef module_fixture(*, legacy=True):\n    source = pinned.SOLVER_PATH.read_text()\n    tree = ast.parse(source)\n    methods = {}\n    for cls in tree.body:\n        if isinstance(cls, ast.ClassDef):\n            for method in cls.body:\n                if isinstance(method, (ast.FunctionDef, ast.AsyncFunctionDef)):\n                    if method.name in {"_finish_if_needed", "_finish_remaining", "_run_games", "_drain_game_tasks"}:\n                        methods[method.name] = textwrap.dedent(ast.get_source_segment(source, method))\n    if legacy:\n        lines = methods["_finish_if_needed"].splitlines()\n        cut = next(i for i, line in enumerate(lines) if "# Actions bill" in line)\n        methods["_finish_if_needed"] = "\\n".join(lines[:cut]) + "\\n        self.game.finish_game()\\n"\n    temp = tempfile.TemporaryDirectory(prefix="arc3-final-tail-cpu-")\n    file = Path(temp.name) / "public_finalizers.py"\n    code = "from __future__ import annotations\\n\\n" + "\\n\\n".join(methods.values()) + "\\n"\n    file.write_text(code)\n    scope = dict(asyncio=asyncio, functools=functools,\n                 _analyzer_reported_tokens=pinned._solver["_analyzer_reported_tokens"])\n    exec(compile(code, str(file), "exec"), scope)\n    session_cls = type("_HarnessGameSession", (), {"_finish_if_needed": scope["_finish_if_needed"]})\n    solver_cls = type("HarnessSolver", (), {k: v for k, v in scope.items()\n                      if k in {"_finish_remaining", "_run_games", "_drain_game_tasks"}})\n    module = SimpleNamespace(_HarnessGameSession=session_cls, HarnessSolver=solver_cls,\n                             _analyzer_reported_tokens=scope["_analyzer_reported_tokens"])\n    module._fixture_temp = temp\n    return module\n\n\nclass TailAccountingTests(unittest.TestCase):\n    def setUp(self):\n        self.module = module_fixture()\n        self.addCleanup(self.module._fixture_temp.cleanup)\n        self.assertTrue(install(self.module))\n\n    def finish(self, subject):\n        with redirect_stdout(io.StringIO()):\n            self.module._HarnessGameSession._finish_if_needed(subject)\n\n    def test_normal_tail_and_history_score_preserved(self):\n        subject = pinned.session()\n        run = subject.game.game_run\n        record = pinned._game["ActionRecord"](\n            action=SimpleNamespace(id="ACTION1", data={}), generated_tokens=100,\n            uncached_input_tokens=0, wallclock_seconds=.1)\n        run.history.append(record)\n        run.actions_per_level[0] = 1\n        run.levels_completed = 1\n        history = run.history\n        self.finish(subject)\n        run = subject.game.game_run\n        self.assertEqual(run.final_generated_tokens, 50)\n        self.assertIs(run.history, history)\n        self.assertEqual(run.actions_per_level, [1, 0])\n        self.assertEqual(run.history, [record])\n        self.assertEqual(sum(r.generated_tokens for r in run.history) + run.final_generated_tokens, 150)\n        self.assertEqual(run.final_score, run._compute_final_score())\n        self.assertEqual(subject.game.close_count, 1)\n\n    def test_normal_finish_stays_idempotent_if_counter_advances(self):\n        subject = pinned.session()\n        self.finish(subject)\n        subject.analyzer.generated_tokens = 500\n        self.finish(subject)\n        self.assertEqual(subject.game.game_run.final_generated_tokens, 50)\n        self.assertEqual(subject.token_baseline, 150)\n        self.assertEqual(subject.game.close_count, 1)\n\n    def test_cancelled_and_crashed_normal_finish(self):\n        for state, stopped, expected in [("playing", True, "cancelled"), ("crashed", False, "crashed")]:\n            subject = pinned.session(state=state, stopped=stopped)\n            self.finish(subject)\n            self.assertEqual(subject.game.game_run.state, expected)\n            self.assertEqual(subject.game.game_run.final_generated_tokens, 50)\n\n    def test_legacy_counter_and_counter_reset(self):\n        subject = pinned.session()\n        subject.analyzer = SimpleNamespace(total_tokens=170)\n        self.finish(subject)\n        self.assertEqual(subject.game.game_run.final_generated_tokens, 70)\n        subject = pinned.session(generated=20)\n        self.finish(subject)\n        self.assertEqual(subject.game.game_run.final_generated_tokens, 0)\n\n    def test_no_run_and_previously_finished_are_not_changed(self):\n        subject = pinned.session()\n        subject.game.game_run = None\n        self.finish(subject)\n        self.assertEqual(subject.game.close_count, 0)\n        subject = pinned.session(stopped=True)\n        run = subject.game.game_run\n        run.final_score, run.final_generated_tokens, run.state = 0.0, 17, "cancelled"\n        self.finish(subject)\n        self.assertEqual(run.final_generated_tokens, 17)\n        self.assertEqual(subject.game.close_count, 0)\n\n    def test_external_close_reconciles_once_preserving_score_state_and_close_count(self):\n        subject = pinned.session(stopped=True)\n        host = self.module.HarnessSolver()\n        host._stop_event = subject.stop_event\n        with redirect_stdout(io.StringIO()):\n            host._finish_remaining([subject.game])\n        run = subject.game.game_run\n        score, state, history = run.final_score, run.state, list(run.history)\n        self.assertTrue(run._arc3_external_tail_pending)\n        self.finish(subject)\n        self.assertFalse(run._arc3_external_tail_pending)\n        self.assertEqual(run.final_generated_tokens, 50)\n        self.assertEqual((run.final_score, run.state, run.history), (score, state, history))\n        subject.analyzer.generated_tokens = 300\n        self.finish(subject)\n        self.assertEqual(run.final_generated_tokens, 50)\n        self.assertEqual(subject.game.close_count, 1)\n\n    def test_already_finished_run_never_gets_external_marker(self):\n        subject = pinned.session(stopped=True)\n        self.finish(subject)\n        host = self.module.HarnessSolver()\n        host._stop_event = subject.stop_event\n        host._finish_remaining([subject.game])\n        self.assertFalse(getattr(subject.game.game_run, "_arc3_external_tail_pending", False))\n\n    def test_actual_async_cancellation_drain_then_late_worker(self):\n        async def scenario():\n            host = self.module.HarnessSolver()\n            host._stop_event = threading.Event()\n            host.concurrency = 1\n            host.cancel_drain_timeout_s = .01\n            host._worker_pool = ThreadPoolExecutor(max_workers=1)\n            host._local_server_for_game_index = lambda _: None\n            subject = pinned.session()\n            subject.stop_event = host._stop_event\n            entered, release = threading.Event(), threading.Event()\n            def play_one(*args):\n                entered.set()\n                release.wait(timeout=2)\n                self.finish(subject)\n            host._play_one = play_one\n            task = asyncio.create_task(host._run_games([subject.game]))\n            try:\n                for _ in range(1000):\n                    if entered.is_set():\n                        break\n                    await asyncio.sleep(.001)\n                self.assertTrue(entered.is_set())\n                task.cancel()\n                with redirect_stdout(io.StringIO()):\n                    with self.assertRaises(asyncio.CancelledError):\n                        await task\n                self.assertEqual(subject.game.game_run.final_generated_tokens, 0)\n                release.set()\n            finally:\n                release.set()\n                host._worker_pool.shutdown(wait=True)\n            self.assertEqual(subject.game.game_run.final_generated_tokens, 50)\n            self.assertEqual(subject.game.game_run.state, "cancelled")\n            self.assertEqual(subject.game.close_count, 1)\n        asyncio.run(scenario())\n\n    def test_fail_closed_and_install_idempotence(self):\n        self.assertFalse(install(self.module))\n        bad = module_fixture()\n        self.addCleanup(bad._fixture_temp.cleanup)\n        bad._HarnessGameSession._finish_if_needed = lambda self: None\n        with self.assertRaises(RuntimeError):\n            install(bad)\n        self.assertFalse(getattr(bad, "_arc3_final_tail_accounting_installed", False))\n\n    def test_accepts_existing_basic_tail_patch(self):\n        patched = module_fixture(legacy=False)\n        self.addCleanup(patched._fixture_temp.cleanup)\n        self.assertTrue(install(patched))\n\n    def test_foreign_wraps_does_not_bypass_source_guard(self):\n        for owner_name, method_name in [("_HarnessGameSession", "_finish_if_needed"),\n                                        ("HarnessSolver", "_finish_remaining")]:\n            foreign = module_fixture()\n            self.addCleanup(foreign._fixture_temp.cleanup)\n            owner = getattr(foreign, owner_name)\n            original = getattr(owner, method_name)\n            @functools.wraps(original)\n            def changed(*args, **kwargs):\n                raise AssertionError("foreign behavior")\n            setattr(owner, method_name, changed)\n            with self.assertRaises(RuntimeError):\n                install(foreign)\n            self.assertIs(getattr(owner, method_name), changed)\n\n'
assert hashlib.sha256(HELPER_SOURCE.encode()).hexdigest() == PROVENANCE['helper_sha256']
assert hashlib.sha256(SOLVER_SOURCE.encode()).hexdigest() == PROVENANCE['embedded_solver_excerpt_sha256']
assert hashlib.sha256(GAME_SOURCE.encode()).hexdigest() == PROVENANCE['embedded_game_excerpt_sha256']
print(json.dumps(PROVENANCE, indent=2))


{
  "public_fixture_repository": "https://github.com/acco-cyber/ARC-AGI-3",
  "public_fixture_commit": "58a67ca1c7ddf0d516c7a9376d9f913f9ff5bc60",
  "upstream": "https://github.com/Tufalabs/duck-harness",
  "solver_fixture_sha256": "e22d1425b6dfe5143f6984d8ccfd21fd4435dccf2859a6985d92d985cb91c14e",
  "game_fixture_sha256": "8609e0646e96616decc936daf4bff102afd480beb69ef1e9e778cd2c1c7df225",
  "embedded_solver_excerpt_sha256": "1989eee41a17dad5801a791b53dc6e7b57b4efe2b61f66bc968dd43152f57e79",
  "embedded_game_excerpt_sha256": "4caf364fb04ac5a15d8b1ab3c6cfe30639069dc89251955e945bf3ceede98554",
  "helper_sha256": "ccf3813d64853ab893eab7c763eb3a13774204d33bb39ff3d3cb9c04968c93f5"
}


## Isolated public fixtures and regression suite

The temporary source files are necessary for source inspection and fail-closed installation guards. No GPU dependencies are imported. Every test creates a fresh pair of finalizer classes. The cancellation test uses the actual `_run_games` and `_drain_game_tasks` methods with a 0.01-second test drain instead of the production 120-second drain; Events hold the worker until external cancellation has finished.


In [2]:
_temp = tempfile.TemporaryDirectory(prefix='arc3-tail-notebook-')
_embedded_solver_path = Path(_temp.name) / 'public_solver_excerpt.py'
_embedded_game_path = Path(_temp.name) / 'public_game_excerpt.py'
_embedded_solver_path.write_text(SOLVER_SOURCE)
_embedded_game_path.write_text(GAME_SOURCE)
_helper = types.ModuleType('final_tail_accounting')
sys.modules[_helper.__name__] = _helper
exec(compile(HELPER_SOURCE, 'embedded_final_tail_accounting.py', 'exec'), _helper.__dict__)
_pinned = types.ModuleType('test_accounting')
_pinned.__dict__.update(_embedded_solver_path=_embedded_solver_path, _embedded_game_path=_embedded_game_path)
sys.modules[_pinned.__name__] = _pinned
exec(compile(FIXTURE_SOURCE, 'embedded_public_fixture_support.py', 'exec'), _pinned.__dict__)
_tests = types.ModuleType('arc3_tail_cpu_tests')
sys.modules[_tests.__name__] = _tests
exec(compile(TEST_SOURCE, 'embedded_tail_tests.py', 'exec'), _tests.__dict__)
_suite = unittest.defaultTestLoader.loadTestsFromTestCase(_tests.TailAccountingTests)
_result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(_suite)
assert _result.wasSuccessful(), 'CPU accounting regressions failed'


test_accepts_existing_basic_tail_patch (arc3_tail_cpu_tests.TailAccountingTests.test_accepts_existing_basic_tail_patch) ... ok
test_actual_async_cancellation_drain_then_late_worker (arc3_tail_cpu_tests.TailAccountingTests.test_actual_async_cancellation_drain_then_late_worker) ... ok
test_already_finished_run_never_gets_external_marker (arc3_tail_cpu_tests.TailAccountingTests.test_already_finished_run_never_gets_external_marker) ... ok
test_cancelled_and_crashed_normal_finish (arc3_tail_cpu_tests.TailAccountingTests.test_cancelled_and_crashed_normal_finish) ... ok
test_external_close_reconciles_once_preserving_score_state_and_close_count (arc3_tail_cpu_tests.TailAccountingTests.test_external_close_reconciles_once_preserving_score_state_and_close_count) ... ok
test_fail_closed_and_install_idempotence (arc3_tail_cpu_tests.TailAccountingTests.test_fail_closed_and_install_idempotence) ... ok
test_foreign_wraps_does_not_bypass_source_guard (arc3_tail_cpu_tests.TailAccountingTests.test_foreig

## Result and integration boundary

Normal, cancelled and crashed finalization preserve unbilled tail tokens. Only cancellation runs explicitly marked by this installer can be reconciled after external finalization; ordinary completion remains idempotent if the analyzer counter later advances. The late reconciliation changes token diagnostics and their final timing, while preserving the game score, state, action history and environment close count.

**Limits:** if the worker never returns, this installer cannot recover its final counter. A benchmark serialized before the worker returns must be saved again afterward to contain the late update. Only the two pinned source shapes are supported; this is intentionally not a general monkeypatch for unknown future code. The public test fixture does not qualify a model-serving workload or a hidden competition score.

For a separately reviewed future integration, run `install(actual_solver_module)` after loading the pinned solver and before any game starts. The notebook does not perform that integration or modify any existing scoring notebook.


In [3]:
summary = {
    'tests_run': _result.testsRun,
    'failures': len(_result.failures),
    'errors': len(_result.errors),
    'passed': _result.wasSuccessful(),
    'gpu_operations': 0,
    'network_requests': 0,
    'model_loads': 0,
    'score_evidence': False,
    'scoring_notebook_modified': False,
    'late_worker_recovery_requires_worker_return_and_resave': True,
}
print(json.dumps(summary, indent=2))
_temp.cleanup()


{
  "tests_run": 11,
  "failures": 0,
  "errors": 0,
  "passed": true,
  "gpu_operations": 0,
  "network_requests": 0,
  "model_loads": 0,
  "score_evidence": false,
  "scoring_notebook_modified": false,
  "late_worker_recovery_requires_worker_return_and_resave": true
}
